# Dataset preprocessing

In [1]:
def default_params(): 
    return {
        'setting': 'strict', #lenient, #strict
        'model_results': {
                'M1' : '/workspaces/mica/repos/pii_final/casual_dataset/3b_stablecode_stack_v2_fim_casual_dataset.json',
                'M2' : '/workspaces/mica/repos/pii_final/casual_dataset/3b_qwen_stack_v2_fim_casual_dataset.json',
                'M3' : '/workspaces/mica/repos/pii_final/casual_dataset/7b_codellama_stack_v2_fim_casual_dataset.json', 
                'M4' : '/workspaces/mica/repos/pii_final/casual_dataset/7b_qwen_stack_v2_fim_casual_dataset.json',
                'M5' : '/workspaces/mica/repos/pii_final/casual_dataset/13b_codellama_stack_v2_fim_casual_dataset.json', 
                'M6' : '/workspaces/mica/repos/pii_final/casual_dataset/14b_qwen_stack_v2_fim_casual_dataset.json',
             }, 
        'output_dir': '/workspaces/mica/datax/causal_revision_out',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
    }
params = default_params()

In [2]:
models = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']

In [3]:
threshold_presets = {
    "balanced": {
        "conf_high_q": 0.75,
        "conf_low_q": 0.25,
        "var_high_q": 0.75,
    },
    "lenient": {
        "conf_high_q": 0.67,
        "conf_low_q": 0.33,
        "var_high_q": 0.67,
    },
    "strict": {
        "conf_high_q": 0.80,
        "conf_low_q": 0.20,
        "var_high_q": 0.80,
    },
}

### Imports

In [4]:
import os 
import pandas as pd
import lizard
import re
from tree_sitter import Language, Parser
import tree_sitter_java as tsjava
import io
import sys
from pathlib import Path
from itertools import chain

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Analysis

In [6]:
class TreeSitterManager():
    def __init__(self, lang):
        #self.language = self.get_language(lang)
        self.language = lang
        self.parser = Parser(self.language)
        #self.parser.set_language(self.language)

    def get_ast_errors_and_deep(self, code):
        node_tree = self.parser.parse(bytes(code, "utf8"))
        return self.__detect_ast_errors_and_deep(node_tree.root_node, set())

    def __detect_ast_errors_and_deep(self, node_root, identifier_set = set(), level=0, max_level=0, count = 0):
        """Traverses the tree catch errors and evaluate tree levels"""
        # if not node_root.has_error:
        #     return [], 0
        counter = node_root.child_count

        results = []
        if node_root.type == "ERROR":
            results.append(node_root.text.decode("utf-8"))
        elif node_root.type == "identifier":
            identifier_set.add(node_root.text)
        level += 1
        for n in node_root.children:
            x, identifier_set, y, max_level, count = self.__detect_ast_errors_and_deep(n, identifier_set, level, max_level)
            max_level = max(y, max_level)
            counter += count
            if len(x) > 0:
                results.extend(x)

        return results, identifier_set, max_level, level, counter

In [7]:
def analyze_method_java(code_string):
    temp_file_dir = f"{params['cache_dir']}/mica/lizard"
    create_folder(temp_file_dir)
    temp_file_path = f"{temp_file_dir}/temp.java"
    
    # Redirect stdout to suppress Lizard's prints
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    try:
        analysis = lizard.analyze_file.analyze_source_code(temp_file_path, code_string)
    finally:
        sys.stdout = old_stdout  # Restore stdout

    # Extract function details
    functions = []
    for function in analysis.function_list:
        functions.append({
            "fun_name": function.name,
            "complexity": function.cyclomatic_complexity,
            "nloc": function.nloc,
            "token_counts": function.token_count
        })
    
    return functions[0] if functions else None


In [8]:
def append_covariates(code_df, code_column, variable_type):
    ast_error_detector = TreeSitterManager(Language(tsjava.language()))

    def compute_covariates(row):
        java_code = row[code_column]
        lizard_results = analyze_method_java(java_code)
        try:
            ast_errors, identifier_set, ast_deep, level, count = ast_error_detector.get_ast_errors_and_deep(java_code)
        except RecursionError as e:
            print(f"RecursionError in AST analysis: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values
        except Exception as e:
            print(f"Error computing AST errors: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values

        return pd.Series({
            variable_type+'_complexity': lizard_results.get('complexity') if lizard_results else None,
            variable_type+'_nloc': lizard_results.get('nloc') if lizard_results else None,
            variable_type+'_token_counts': lizard_results.get('token_counts') if lizard_results else None,
            variable_type+'_ast_levels': ast_deep,
            variable_type+'_n_ast_nodes': count,
            variable_type+'_n_ast_errors': len(ast_errors) if ast_errors else 0,
            variable_type+'_ast_errors': ast_errors,
            variable_type+'_n_identifiers': len(identifier_set) if identifier_set else 0,
            variable_type+'_identifiers': identifier_set
        })
    covarites_df = code_df.apply(lambda row: compute_covariates(row), axis=1)
    code_df = pd.concat([code_df, covarites_df], axis=1)
    return code_df

In [9]:
def format_java_code(java_code: str) -> str:
    # Remove <s> and </s> tags
    java_code = re.sub(r"</?s>", "", java_code)
    
    # Fix spaces around dots in package and import statements
    java_code = re.sub(r"\s*\.\s*", ".", java_code)
    
    # Fix multiple spaces to single space
    java_code = re.sub(r"\s+", " ", java_code)
    
    # Ensure proper line breaks after semicolons and braces
    java_code = re.sub(r";\s*", ";\n", java_code)
    java_code = re.sub(r"\{\s*", "{\n", java_code)
    java_code = re.sub(r"\}\s*", "}\n", java_code)
    
    # Ensure proper indentation for better readability
    formatted_lines = []
    indent_level = 0
    for line in java_code.split("\n"):
        stripped = line.strip()
        if stripped.endswith("}"):
            indent_level -= 1

        formatted_lines.append("    " * max(indent_level, 0) + stripped)

        if stripped.endswith("{"):
            indent_level += 1

    return "\n".join(formatted_lines).strip()

In [10]:
def format_java_code_bk(java_code: str):
    return java_code

### Execute

In [11]:
def extract_rule_label(path):
    match = re.search(r'rule_\w+', path)  # Capture 'rule_' followed by one or more word characters
    return match.group(0) if match else None

In [12]:
def compute_relative_score(actual_value, all_values, epsilon=1e-15):
    relative_value = (actual_value - min(all_values)) / (max(all_values) - min(all_values) + epsilon)
    return relative_value 

In [13]:
def compute_relative_outcome_scores_per_setting(s_file_member, s_file_non_member, outcomes):
    s_file_member = s_file_member.copy()
    
    for outcome in outcomes:
        combined_outcome_values = pd.concat([s_file_member[outcome], s_file_non_member[outcome]], ignore_index=True)
        s_file_member[f'relative_{outcome}'] = s_file_member[outcome].apply(lambda x: compute_relative_score(x, combined_outcome_values))

    return s_file_member


In [14]:
def count_tokens_by_space(text: str) -> int:
    """
    Returns the number of tokens in the input string.
    Tokens are defined as whitespace-separated words.
    """
    return len(text.split())

In [15]:
def classify_sample(confidence, variability,
                      conf_high_q=0.75, conf_low_q=0.25, var_high_q=0.75,
                      conf_all=None, var_all=None):
    """
    Quantile-based classification. Provide arrays conf_all/var_all from your dataset
    to set thresholds from empirical quantiles.
    Returns one of {"easy", "ambiguous", "hard"}.
    """
    if conf_all is None or var_all is None:
        raise ValueError("Pass conf_all and var_all to compute quantile thresholds.")

    import numpy as np
    conf_hi = np.quantile(conf_all, conf_high_q)
    conf_lo = np.quantile(conf_all, conf_low_q)
    var_hi  = np.quantile(var_all,  var_high_q)
    var_lo  = np.quantile(var_all,  1 - var_high_q)  # symmetric low cutoff

    if variability >= var_hi:
        return "ambiguous"
    if variability <= var_lo and confidence >= conf_hi:
        return "easy"
    if variability <= var_lo and confidence <= conf_lo:
        return "hard"
    # in-between: fall back to whichever axis is more decisive
    return "ambiguous" if variability > (var_hi + var_lo) / 2 else ("easy" if confidence >= (conf_hi + conf_lo)/2 else "hard")



In [16]:
def process_dataframe(df, code_column):
        df['n_tokens'] = df['text'].apply(count_tokens_by_space)
        #df['type'] = df.apply(lambda row: classify_sample(confidence=row['confidence'], variability=row['variability'], conf_all=df['confidence'], var_all=df['variability']), axis=1)
        df[code_column] = df[code_column].map(format_java_code)
        df = append_covariates(df, code_column, 't')
        #df['model'] = params['current_model']
        df['outcome'] = df['attack_hit'].apply(lambda x: 1 if x else 0)
        return df

In [17]:
def classify_dataframe(df, model_id):
        df['type'] = df.apply(lambda row: classify_sample(confidence=row['confidence'], variability=row['variability'], conf_all=df['confidence'], var_all=df['variability'], **threshold_presets[params['setting']]), axis=1)
        df['model'] = model_id
        return df

In [18]:
for model_id in models:
    pii_results_df = process_dataframe(pd.read_json(params['model_results'][model_id]), 'text')
    pii_results_df = classify_dataframe(pii_results_df, model_id)
    pii_results_df.reset_index(drop=True)
    print(f"===================== SAVING DATA FOR {model_id} ==================")
    output_path = f"{params['output_dir']}/{params['setting']}"
    create_folder(output_path)
    pii_results_df.to_json(f"{output_path}/{model_id}.json", orient="records", lines=False, indent=4)

===================== SAVING DATA FOR M1 ==================
===================== SAVING DATA FOR M2 ==================
===================== SAVING DATA FOR M3 ==================
===================== SAVING DATA FOR M4 ==================
===================== SAVING DATA FOR M5 ==================
===================== SAVING DATA FOR M6 ==================


### Storing data

In [19]:
#print(f"===================== SAVING DATA FOR {params['current_model']} ==================m ")
#output_path = f"{params['output_dir']}/{params['setting']}"
#create_folder(output_path)
#pii_results_df.to_json(f"{output_path}/{params['current_model']}.json", orient="records", lines=False, indent=4)